# Integritate si profilarea cohortului

**Scop:** verificarea calitatii datelor, distributia claselor pe subiecti si sesiuni, statistici separate pentru canalele EEG vs auxiliare (`ECG`, `EOG`) si explorarea grafica a semnalelor.

**Input-uri:** `data/manifest.parquet`, `data/data_sub_*.parquet`, `data/data_sub_*.npz`, `config.py`.

**Output-uri:** tabele CSV si raport Markdown in `reports/metrics/`, plus grafice in `reports/figures/eda/`.

**Regula de memorie:** incarcam **un singur subiect pe rand** si eliberam memoria (`del`, `gc.collect()`) dupa fiecare pas.

# 2.1 Pregatirea mediului

Importam librariile necesare, incarcam `config.py` si setam pragurile simple de verificare.

In [ ]:
%pip install -r ../requirements.txt

In [1]:
# Import necessary libraries

import gc
import sys
import dask
import numpy as np
import pandas as pd
import seaborn as sns
import warnings as wn
import dask.dataframe as dd
import matplotlib.pyplot as plt

from pathlib import Path

In [2]:
# Ignore all warnings
wn.filterwarnings("ignore")

In [3]:
# Import configuration module

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

import config as cfg
cfg.ensure_directories()

In [28]:
# Create simple settings and folders

FLATLINE_THRESHOLD = 1e-8
EXTREME_VAR_MULTIPLIER = 5.0
LOW_VAR_MULTIPLIER = 0.1
SEED = cfg.RANDOM_SEED

rng = np.random.default_rng(SEED)
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

# 2.2 Incarcarea manifestului si lista de subiecti

Citim `data/manifest.parquet` generat in notebook-ul de achizitie de date si pastram doar subiectii utilizabili si care indeplinesc conditia `usable == True`.

In [8]:
# Load manifest and get usable subject IDs

manifest = pd.read_parquet(cfg.MANIFEST_PATH)
subjects = sorted(manifest[manifest["usable"]]["subject_id"].unique().tolist())

In [10]:
# Usable subjects in the storage

print("Subiecti utilizabili:", subjects)
manifest[["subject_id", "format", "n_trials", "n_channels", "n_samples", "sfreq", "sessions", "usable"]]

Subiecti utilizabili: [1, 2, 3]


,subject_id,format,n_trials,n_channels,n_samples,sfreq,sessions,usable
0,1,npz,600,64,4001,1000.0,"0,1,2",True
1,1,parquet,600,64,4001,1000.0,"0,1,2",True
2,2,npz,600,64,4001,1000.0,"0,1,2",True
3,2,parquet,600,64,4001,1000.0,"0,1,2",True
4,3,npz,600,64,4001,1000.0,"0,1,2",True
5,3,parquet,600,64,4001,1000.0,"0,1,2",True


# 2.3 Audit simplu pe Parquet cu Dask

Citim pe rand fiecare fisier Parquet cu Dask, redenumim coloanele `ch_*` cu `cfg.CHANNEL_MAP` si verificam valorile lipsa si limitele semnalului.

In [ ]:
# Simple function to check one Parquet file with Dask

def check_parquet(sub_id):
    pq_path = cfg.get_subject_parquet_path(sub_id)
    ddf = dd.read_parquet(pq_path, engine=cfg.PARQUET_ENGINE)
    ddf = ddf.rename(columns=cfg.CHANNEL_MAP)
    aux_cols = cfg.ECG_CHANNELS + cfg.EOG_CHANNELS

    n_rows, nulls, n_trials, ts_min, ts_max, eeg_min, eeg_max, aux_min, aux_max = dask.compute(
        ddf.shape[0],
        ddf.isnull().sum().sum(),
        ddf["trial"].nunique(),
        ddf["timestamp"].min(),
        ddf["timestamp"].max(),
        ddf[cfg.EEG_CHANNELS].min().min(),
        ddf[cfg.EEG_CHANNELS].max().max(),
        ddf[aux_cols].min().min(),
        ddf[aux_cols].max().max(),
    )

    n_samples = int(ts_max) - int(ts_min) + 1
    
    return {
        "subject_id": int(sub_id),
        "pq_rows": int(n_rows),
        "pq_trials": int(n_trials),
        "pq_samples": n_samples,
        "rows_ok": int(n_rows) == int(n_trials) * n_samples,
        "nulls": int(nulls),
        "eeg_min": float(eeg_min),
        "eeg_max": float(eeg_max),
        "aux_min": float(aux_min),
        "aux_max": float(aux_max),
    }

In [12]:
# Run Parquet checks for each subject

pq_rows = []
for sub_id in subjects:
    info = check_parquet(sub_id)
    pq_rows.append(info)
    print(f"Subiect {sub_id:02d}: randuri={info['pq_rows']}, nulls={info['nulls']}, EEG=[{info['eeg_min']:.4f}, {info['eeg_max']:.4f}], AUX=[{info['aux_min']:.1f}, {info['aux_max']:.1f}]")

pq_df = pd.DataFrame(pq_rows)

Subiect 01: randuri=2400600, nulls=0, EEG=[-0.1649, 0.1869], AUX=[-375.0, 185.3]
Subiect 02: randuri=2400600, nulls=0, EEG=[-0.1991, 0.1927], AUX=[-375.0, 226.5]
Subiect 03: randuri=2400600, nulls=0, EEG=[-0.2002, 0.1489], AUX=[-375.0, 262.5]


In [13]:
# Show the parquet dataframe
pq_df

,subject_id,pq_rows,pq_trials,pq_samples,rows_ok,nulls,eeg_min,eeg_max,aux_min,aux_max
0,1,2400600,600,4001,True,0,-0.164860,0.186856,-375.0,185.344437
1,2,2400600,600,4001,True,0,-0.199141,0.192733,-375.0,226.498108
2,3,2400600,600,4001,True,0,-0.200235,0.148855,-375.0,262.516296


# 2.4 Distributia claselor si sesiunilor (NPZ)

Citim doar etichetele `y` si `metadata` (`subject`, `session`, `run`) din fisierele NPZ pentru a numara trialurile reale pe fiecare sesiune si clasa.

In [16]:
# Count observed trials per subject, session, and class

class_rows = []
trial_rows = []

for sub_id in subjects:
    data = np.load(cfg.get_subject_npz_path(sub_id), allow_pickle=True)
    y = data["y"].astype(str)
    meta = data["metadata"]
    sessions = [str(row[1]) for row in meta]

    df_sub = pd.DataFrame({"subject_id": sub_id, "session": sessions, "label": y})
    counts = df_sub.groupby(["subject_id", "session", "label"]).size().reset_index(name="n_trials")
    class_rows.append(counts)

    pq_trials = int(pq_df.loc[pq_df["subject_id"] == sub_id, "pq_trials"].iloc[0])
    trial_rows.append({
        "subject_id": sub_id,
        "npz_trials": len(y),
        "n_sessions": len(set(sessions)),
        "sessions": ",".join(sorted(set(sessions))),
        "match_parquet": len(y) == pq_trials,
    })
    print(f"Subiect {sub_id:02d}: {len(y)} trialuri, sesiuni={sorted(set(sessions))}, coincide_parquet={len(y) == pq_trials}")

    del data, y, meta, df_sub
    gc.collect()

Subiect 01: 600 trialuri, sesiuni=['0', '1', '2'], coincide_parquet=True
Subiect 02: 600 trialuri, sesiuni=['0', '1', '2'], coincide_parquet=True
Subiect 03: 600 trialuri, sesiuni=['0', '1', '2'], coincide_parquet=True


In [17]:
# Show the class 

class_df = pd.concat(class_rows, ignore_index=True)
trial_df = pd.DataFrame(trial_rows)

In [19]:
# Show class counts table per subject and session
class_df.pivot_table(index=["subject_id", "session"], columns="label", values="n_trials", fill_value=0)

label               left_hand  right_hand
subject_id session                       
1          0            100.0       100.0
           1            100.0       100.0
           2            100.0       100.0
2          0            100.0       100.0
           1            100.0       100.0
           2            100.0       100.0
3          0            100.0       100.0
           1            100.0       100.0
           2            100.0       100.0

# 2.5 Statistici per canal (EEG vs Auxiliare) si anomalii

Calculam statisticile pe fiecare canal (`mean`, `std`, `var`, `min`, `p05`, `p50`, `p95`, `max`, `range`) si verificam valorile NaN/Inf, canalele flatline si canalele EEG cu varianta extrema.

In [20]:
# Simple helper to assign a scalp region to each channel

def get_region(ch_name):
    if ch_name in cfg.ECG_CHANNELS or ch_name in cfg.EOG_CHANNELS:
        return "Aux"
    if ch_name.startswith(("Fp", "AF", "F")) and not ch_name.startswith(("FC", "FT")):
        return "Frontal"
    if ch_name.startswith(("FC", "FT", "C", "T")) and not ch_name.startswith(("CP", "TP")):
        return "Central/Motor"
    if ch_name.startswith(("CP", "TP", "P")):
        return "Parietal"
    return "Occipital"

In [22]:
# Simple function to compute channel stats for one subject

def get_channel_stats(sub_id):
    data = np.load(cfg.get_subject_npz_path(sub_id), allow_pickle=True)
    X = data["X"]

    n_nan = int(np.isnan(X).sum())
    n_inf = int(np.isinf(X).sum())
    rows = []

    for i, ch_name in enumerate(cfg.MED_COLUMNS):
        vals = X[:, i, :]
        p05, p50, p95 = np.percentile(vals, [5, 50, 95])
        ch_std = float(np.std(vals))
        ch_min = float(np.min(vals))
        ch_max = float(np.max(vals))
        flat_trials = int((vals.std(axis=1) <= FLATLINE_THRESHOLD).sum())

        rows.append({
            "subject_id": sub_id,
            "channel": ch_name,
            "type": cfg.CHANNEL_TYPES[ch_name],
            "region": get_region(ch_name),
            "mean": float(np.mean(vals)),
            "std": ch_std,
            "var": float(np.var(vals)),
            "min": ch_min,
            "p05": float(p05),
            "p50": float(p50),
            "p95": float(p95),
            "max": ch_max,
            "range": ch_max - ch_min,
            "flatline_trials": flat_trials,
            "is_flatline": ch_std <= FLATLINE_THRESHOLD,
        })

    del data, X
    gc.collect()

    df_ch = pd.DataFrame(rows)
    eeg_median_std = df_ch[df_ch["type"] == "eeg"]["std"].median()
    df_ch["is_extreme_var"] = (
        (df_ch["type"] == "eeg")
        & ((df_ch["std"] > EXTREME_VAR_MULTIPLIER * eeg_median_std) | (df_ch["std"] < LOW_VAR_MULTIPLIER * eeg_median_std))
    )
    
    return df_ch, n_nan, n_inf

In [23]:
# Run channel stats loop across all usable subjects

all_ch_dfs = []
quality_rows = []

for sub_id in subjects:
    df_ch, n_nan, n_inf = get_channel_stats(sub_id)
    all_ch_dfs.append(df_ch)
    
    quality_rows.append({
        "subject_id": sub_id,
        "n_nan": n_nan,
        "n_inf": n_inf,
        "flatline_channels": int(df_ch["is_flatline"].sum()),
        "partial_flatline_channels": int((df_ch["flatline_trials"] > 0).sum()),
        "extreme_var_eeg": int(df_ch["is_extreme_var"].sum()),
    })
    
    print(f"Subiect {sub_id:02d}: NaN={n_nan}, Inf={n_inf}, flatline={df_ch['is_flatline'].sum()}, partial_flat={int((df_ch['flatline_trials'] > 0).sum())}, extreme_eeg={df_ch['is_extreme_var'].sum()}")

channel_df = pd.concat(all_ch_dfs, ignore_index=True)
quality_df = pd.DataFrame(quality_rows)

Subiect 01: NaN=0, Inf=0, flatline=0, partial_flat=2, extreme_eeg=0
Subiect 02: NaN=0, Inf=0, flatline=0, partial_flat=2, extreme_eeg=0
Subiect 03: NaN=0, Inf=0, flatline=0, partial_flat=4, extreme_eeg=0


In [25]:
# Compare EEG channels vs Auxiliary channels separately
display(channel_df[channel_df["type"] == "eeg"].groupby("subject_id")[["mean", "std", "var", "p05", "p50", "p95", "range"]].mean())

,mean,std,var,p05,p50,p95,range
subject_id,,,,,,,
1,1.561390e-07,0.002456,0.000006,-0.003878,0.000003,0.003870,0.104454
2,-2.625829e-07,0.005768,0.000036,-0.008472,0.000010,0.008471,0.170265
3,-6.971834e-07,0.003944,0.000017,-0.006316,-0.000004,0.006321,0.117641


In [26]:
# Auxilliary ECG / EOG
display(channel_df[channel_df["type"] != "eeg"][["subject_id", "channel", "type", "mean", "std", "min", "max", "flatline_trials"]])

,subject_id,channel,type,mean,std,min,max,flatline_trials
59,1,ECG,ecg,-326.187632,79.029837,-375.000000,-58.441158,400
60,1,EOG1,eog,-182.616658,109.095732,-269.303703,-21.057331,0
61,1,EOG2,eog,-77.972233,58.472500,-138.973595,31.730155,0
62,1,EOG3,eog,41.753678,96.094302,-93.305049,185.344431,0
63,1,EOG4,eog,-201.121464,126.422740,-375.000000,-71.495068,200
123,2,ECG,ecg,-273.927892,146.266708,-375.000000,-37.002663,401
124,2,EOG1,eog,-284.320936,128.621512,-375.000000,-73.546868,400
125,2,EOG2,eog,-84.386909,131.313682,-313.411471,48.629324,0
126,2,EOG3,eog,140.611060,62.297670,46.017478,226.498103,0
127,2,EOG4,eog,68.179260,93.213729,-100.193633,208.005299,0


In [27]:
# Show any flagged channels (flatline, partial flatline, or extreme variance)

flagged_df = channel_df[
    channel_df["is_flatline"] | (channel_df["flatline_trials"] > 0) | channel_df["is_extreme_var"]
][["subject_id", "channel", "type", "std", "flatline_trials", "is_flatline", "is_extreme_var"]]

print(f"Canale semnalate: {len(flagged_df)}")
flagged_df

Canale semnalate: 8


,subject_id,channel,type,std,flatline_trials,is_flatline,is_extreme_var
59,1,ECG,ecg,79.029837,400,False,False
63,1,EOG4,eog,126.422740,200,False,False
123,2,ECG,ecg,146.266708,401,False,False
124,2,EOG1,eog,128.621512,400,False,False
156,3,C3,eeg,0.003397,36,False,False
187,3,ECG,ecg,146.469827,294,False,False
188,3,EOG1,eog,61.412625,400,False,False
191,3,EOG4,eog,92.288146,200,False,False
